# Step 1. Preliminaries

## 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from collections import defaultdict
import torch
import torch.nn as nn

import helpers 

# Configuration
DATA_DIR = 'CamelinaMAGIC-Dataset-Complete'
EXPERIMENTS = [1, 2, 3] # We will process all three
RANDOMIZATION_FILE = os.path.join(DATA_DIR, 'CamelinaMAGIC-ALL-Randomization.xlsx')

print("Libraries imported and configuration set.")

## 1.2 Loading Data and Processing

In [ ]:
# 1.2 Data Loading and Processing


print("Loading data from Experiments:", EXPERIMENTS)

exp_transpiration_dataframes = {} # To store processed dataframes for each experiment

for exp_num in EXPERIMENTS:
    print(f"  Processing Experiment {exp_num}...")
    
    # 1. Define Paths
    transp_file = os.path.join(DATA_DIR, f'CamelinaMAGIC{exp_num}.0-DailyTranspiration.csv')
    weather_file = os.path.join(DATA_DIR, f'CamelinaMAGIC{exp_num}.0-WeatherStation.csv')

    try:
        df_transp = helpers.read_transpiration_csv(transp_file, RANDOMIZATION_FILE, exp_num)
        df_weather = helpers.read_weather_station_csv(weather_file, exp_num)

    except Exception as e:
        print(f"    Skipping Exp {exp_num} due to error: {e}")
        continue

    df_transp = df_transp[df_transp['Treatment'] == 'DR_100']     # Filter for DROUGHT Only Plants
    #helpers.plot_drought_transpiration_df(df_transp, title=f'Experiment {exp_num} - Before VPD Normalization')

    helpers.plot_drought_transpiration_df(df_transp, title=f'Experiment {exp_num} - Before Smoothing')
    #print(f"df_transp before smoothing for Exp {exp_num}:\n", df_transp.head())
    
    df_transp['Transp_original'] = df_transp['Transpiration']  # Keep original for comparison


    # Smooth Data (Rolling Average)

    # cols_to_smooth = ['Weight', 'Transpiration', 'Transp_Norm'] #old line
    cols_to_smooth = ['Transpiration']
    df_transp['Transpiration'] = df_transp.groupby('Sample')[cols_to_smooth].transform(   # Group by Sample to ensure we don't smooth across different plants
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    helpers.plot_drought_transpiration_df(df_transp, title=f'Experiment {exp_num} - After Smoothing')
        
    #print(f"df_transp after smoothing for Exp {exp_num}:\n", df_transp.head())

    exp_transpiration_dataframes[exp_num] = df_transp # Store for later use.

print("Number of dataframes in the dict exp_transpiration_dataframes:", len(exp_transpiration_dataframes))



### Just to visualize important information for now

In [ ]:
combined_df = pd.concat([d for d in exp_transpiration_dataframes.values()], ignore_index=True)

unique_samples = combined_df['Sample'].unique()
print(f"  Found \033[32m\033[1m{len(unique_samples)}\033[0m\033[m unique samples in all the experiments combined:\n ", unique_samples, "\n")

unique_genotypes = combined_df['Genotype'].unique()
print(f"  Found \033[32m\033[1m{len(unique_genotypes)}\033[0m\033[m unique genotypes in all the experiments combined:\n ", unique_genotypes, "\n")


for exp, df in exp_transpiration_dataframes.items():
    unique_exp_samples = df['Sample'].unique()
    print(f"  Found \033[32m\033[1m{len(unique_exp_samples)}\033[0m\033[m unique samples in experiment \033[32m\033[1m{exp}\033[0m\033[m:\n ", unique_exp_samples, "\n")
    
    unique_exp_genotypes = df['Genotype'].unique()
    print(f"  Found \033[32m\033[1m{len(unique_exp_genotypes)}\033[0m\033[m unique genotypes in experiment \033[32m\033[1m{exp}\033[0m\033[m:\n ", unique_exp_genotypes, "\n")


### Prepare Unified Dataset with All Samples (New Approach)

In [ ]:
# Prepare unified dataset: pool all samples across all experiments and genotypes
# Each sample will have: [11 transpiration values, 16 genotype one-hot features, 3 experiment one-hot features]

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np
import pandas as pd

INPUT_WINDOW = list(range(15, 26))   # Days 15-25 inclusive (11 timesteps)
RECOVERY_WINDOW = list(range(36, 41)) # Days 36-40 inclusive

print("Building unified dataset from all experiments and genotypes...")

# Collect all samples with their features and targets
all_samples_data = []

for exp in [1, 2, 3]:
    exp_df = exp_transpiration_dataframes.get(exp)
    if exp_df is None:
        continue
    
    # Group by sample to process each plant individually
    for sample_name, sample_df in exp_df.groupby('Sample'):
        sample_df_sorted = sample_df.sort_values('Days')
        
        # Get input window data (Days 15-25) - SMOOTHED transpiration
        input_data = sample_df_sorted[sample_df_sorted['Days'].isin(INPUT_WINDOW)]
        if len(input_data) < len(INPUT_WINDOW):
            continue  # Skip samples with incomplete input data
        
        # Extract SMOOTHED transpiration values for input window
        transp_features = input_data['Transpiration'].values  # Already smoothed in Cell 5
        
        # Get genotype
        genotype = sample_df['Genotype'].iloc[0]
        
        # Calculate target slope from recovery window (Days 36-40) - SMOOTHED data
        recovery_data = sample_df_sorted[sample_df_sorted['Days'].isin(RECOVERY_WINDOW)]
        if len(recovery_data) < 2:
            continue  # Skip samples without enough recovery data
        
        from scipy.stats import linregress
        target_slope = float(linregress(recovery_data['Days'], recovery_data['Transpiration']).slope)
        
        # Store all information
        all_samples_data.append({
            'Sample': sample_name,
            'Experiment': exp,
            'Genotype': genotype,
            'Transpiration_Features': transp_features,
            'Target_Slope': target_slope
        })

print(f"✓ Collected {len(all_samples_data)} samples total")

# Create feature matrix with one-hot encoded genotypes and experiments
unique_genotypes = sorted(list(set([s['Genotype'] for s in all_samples_data])))
print(f"✓ Found {len(unique_genotypes)} unique genotypes: {unique_genotypes}")

# Build feature matrix WITH ONE-HOT EXPERIMENTS
X_unified = []
y_unified = []
sample_info = []  # Keep track of sample metadata

for sample_data in all_samples_data:
    # Start with SMOOTHED transpiration features (11 values)
    features = list(sample_data['Transpiration_Features'])
    
    # Add one-hot encoded genotype (16 features)
    genotype_encoding = [1 if g == sample_data['Genotype'] else 0 for g in unique_genotypes]
    features.extend(genotype_encoding)
    
    # Add one-hot encoded experiment (3 features)
    exp_encoding = [1 if exp == sample_data['Experiment'] else 0 for exp in [1, 2, 3]]
    features.extend(exp_encoding)
    
    X_unified.append(features)
    y_unified.append(sample_data['Target_Slope'])
    sample_info.append({
        'Sample': sample_data['Sample'],
        'Experiment': sample_data['Experiment'],
        'Genotype': sample_data['Genotype']
    })

X_unified = np.array(X_unified)
y_unified = np.array(y_unified)

print(f"\n{'='*70}")
print(f"UNIFIED DATASET SUMMARY")
print(f"{'='*70}")
print(f"Total samples: {len(X_unified)}")
print(f"Feature dimensions: {X_unified.shape[1]} features")
print(f"  - Transpiration SMOOTHED (Days 15-25): 11 features")
print(f"  - Genotype (one-hot): {len(unique_genotypes)} features")
print(f"  - Experiment (one-hot): 3 features")
print(f"Target: Recovery slope from SMOOTHED data (Days 36-40)")
print(f"Target range: [{y_unified.min():.4f}, {y_unified.max():.4f}]")
print(f"Target mean: {y_unified.mean():.4f}")
print(f"Target std: {y_unified.std():.4f}")
print(f"{'='*70}\n")


In [ ]:
# Train unified models using LOOCV on all samples
# Models: Linear Regression, Random Forest, LSTM
# Each model trains 76 times (once per sample), training on 75 and testing on 1
# Final metrics are computed by averaging performance across all 76 test samples

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import numpy as np
import os
import pickle

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Create models directory
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

print(f"Training unified models with LOOCV on {len(X_unified)} samples...")
print(f"Each fold: Train on {len(X_unified)-1} samples, Test on 1 sample")
print(f"Total folds: {len(X_unified)}\n")

# ============================================================================
# 1. LINEAR REGRESSION
# ============================================================================
print("="*70)
print("LINEAR REGRESSION")
print("="*70)

loo = LeaveOneOut()
lr_preds = []
lr_actuals = []
lr_sample_info = []

for train_idx, test_idx in loo.split(X_unified):
    X_train, X_test = X_unified[train_idx], X_unified[test_idx]
    y_train, y_test = y_unified[train_idx], y_unified[test_idx]
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train Linear Regression model
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)[0]
    
    lr_preds.append(y_pred)
    lr_actuals.append(y_test[0])
    lr_sample_info.append(sample_info[test_idx[0]])

# Calculate metrics by comparing all 76 predictions to actual values
lr_r2 = r2_score(lr_actuals, lr_preds)
lr_mse = mean_squared_error(lr_actuals, lr_preds)
lr_rmse = np.sqrt(lr_mse)
lr_mae = mean_absolute_error(lr_actuals, lr_preds)

print(f"R² Score: {lr_r2:.4f}")
print(f"RMSE: {lr_rmse:.4f}")
print(f"MAE: {lr_mae:.4f}")

# Train final model on all data for deployment
scaler_final_lr = StandardScaler()
X_scaled_final = scaler_final_lr.fit_transform(X_unified)
final_model_lr = LinearRegression()
final_model_lr.fit(X_scaled_final, y_unified)

# Save model
lr_model_path = os.path.join(MODELS_DIR, 'LR_unified_model.pkl')
lr_scaler_path = os.path.join(MODELS_DIR, 'LR_unified_scaler.pkl')
with open(lr_model_path, 'wb') as f:
    pickle.dump(final_model_lr, f)
with open(lr_scaler_path, 'wb') as f:
    pickle.dump(scaler_final_lr, f)
print(f"✓ Saved: {lr_model_path}")
print(f"✓ Saved: {lr_scaler_path}\n")

# ============================================================================
# 2. RANDOM FOREST
# ============================================================================
print("="*70)
print("RANDOM FOREST")
print("="*70)

rf_preds = []
rf_actuals = []

for train_idx, test_idx in loo.split(X_unified):
    X_train, X_test = X_unified[train_idx], X_unified[test_idx]
    y_train, y_test = y_unified[train_idx], y_unified[test_idx]
    
    # Scale features for consistency
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train model
    model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10, min_samples_leaf=2)
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)[0]
    
    rf_preds.append(y_pred)
    rf_actuals.append(y_test[0])

# Calculate metrics by comparing all 76 predictions to actual values
rf_r2 = r2_score(rf_actuals, rf_preds)
rf_mse = mean_squared_error(rf_actuals, rf_preds)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(rf_actuals, rf_preds)

print(f"R² Score: {rf_r2:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"MAE: {rf_mae:.4f}")

# Train final model on all data for deployment
scaler_final_rf = StandardScaler()
X_scaled_final_rf = scaler_final_rf.fit_transform(X_unified)
final_model_rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10, min_samples_leaf=2)
final_model_rf.fit(X_scaled_final_rf, y_unified)

# Save model
rf_model_path = os.path.join(MODELS_DIR, 'RF_unified_model.pkl')
rf_scaler_path = os.path.join(MODELS_DIR, 'RF_unified_scaler.pkl')
with open(rf_model_path, 'wb') as f:
    pickle.dump(final_model_rf, f)
with open(rf_scaler_path, 'wb') as f:
    pickle.dump(scaler_final_rf, f)
print(f"✓ Saved: {rf_model_path}")
print(f"✓ Saved: {rf_scaler_path}\n")

# ============================================================================
# 3. LSTM (PyTorch)
# ============================================================================
print("="*70)
print("LSTM")
print("="*70)

# Define LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, metadata_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        # Combine LSTM output with metadata features
        self.fc1 = nn.Linear(hidden_size + metadata_size, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, output_size)
    
    def forward(self, x_sequence, x_metadata):
        # x_sequence: (batch, seq_len, features) = (batch, 11, 1)
        # x_metadata: (batch, metadata_features) = (batch, 19) for genotype+experiment
        
        lstm_out, _ = self.lstm(x_sequence)
        lstm_out = lstm_out[:, -1, :]  # Take last timestep
        
        # Concatenate with metadata
        combined = torch.cat((lstm_out, x_metadata), dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        output = self.fc2(x)
        return output

# Prepare data for LSTM: separate time-series from metadata
X_sequence = X_unified[:, :11].reshape(-1, 11, 1)  # SMOOTHED Transpiration
X_metadata = X_unified[:, 11:]  # Genotype (16) + Experiment (3) = 19

lstm_preds = []
lstm_actuals = []

print("Training LSTM (this may take a few minutes)...")
for fold_num, (train_idx, test_idx) in enumerate(loo.split(X_unified), 1):
    if fold_num % 10 == 0:
        print(f"  Fold {fold_num}/{len(X_unified)}")
    
    X_seq_train, X_seq_test = X_sequence[train_idx], X_sequence[test_idx]
    X_meta_train, X_meta_test = X_metadata[train_idx], X_metadata[test_idx]
    y_train, y_test = y_unified[train_idx], y_unified[test_idx]
    
    # Convert to PyTorch tensors
    X_seq_train_t = torch.FloatTensor(X_seq_train)
    X_meta_train_t = torch.FloatTensor(X_meta_train)
    y_train_t = torch.FloatTensor(y_train)
    X_seq_test_t = torch.FloatTensor(X_seq_test)
    X_meta_test_t = torch.FloatTensor(X_meta_test)
    
    # Initialize model
    model = LSTMModel(input_size=1, hidden_size=64, num_layers=2, output_size=1, metadata_size=19)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    # Train
    model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        outputs = model(X_seq_train_t, X_meta_train_t)
        loss = criterion(outputs.squeeze(), y_train_t)
        loss.backward()
        optimizer.step()
    
    # Predict
    model.eval()
    with torch.no_grad():
        y_pred = model(X_seq_test_t, X_meta_test_t).item()
    
    lstm_preds.append(y_pred)
    lstm_actuals.append(y_test[0])

# Calculate metrics by comparing all 76 predictions to actual values
lstm_r2 = r2_score(lstm_actuals, lstm_preds)
lstm_mse = mean_squared_error(lstm_actuals, lstm_preds)
lstm_rmse = np.sqrt(lstm_mse)
lstm_mae = mean_absolute_error(lstm_actuals, lstm_preds)

print(f"R² Score: {lstm_r2:.4f}")
print(f"RMSE: {lstm_rmse:.4f}")
print(f"MAE: {lstm_mae:.4f}")

# Train final LSTM model on all data for deployment
print("Training final LSTM model...")
X_seq_all = torch.FloatTensor(X_sequence)
X_meta_all = torch.FloatTensor(X_metadata)
y_all = torch.FloatTensor(y_unified)

final_model_lstm = LSTMModel(input_size=1, hidden_size=64, num_layers=2, output_size=1, metadata_size=19)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(final_model_lstm.parameters(), lr=0.01)

final_model_lstm.train()
for epoch in range(200):
    optimizer.zero_grad()
    outputs = final_model_lstm(X_seq_all, X_meta_all)
    loss = criterion(outputs.squeeze(), y_all)
    loss.backward()
    optimizer.step()

# Save LSTM model
lstm_model_path = os.path.join(MODELS_DIR, 'LSTM_unified_model.pth')
torch.save(final_model_lstm.state_dict(), lstm_model_path)
print(f"✓ Saved: {lstm_model_path}\n")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)

results_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'LSTM'],
    'R²': [lr_r2, rf_r2, lstm_r2],
    'RMSE': [lr_rmse, rf_rmse, lstm_rmse],
    'MAE': [lr_mae, rf_mae, lstm_mae],
    'Samples': [len(X_unified)] * 3,
    'Features': [X_unified.shape[1]] * 3
})

display(results_df.sort_values('R²', ascending=False).reset_index(drop=True))


# Model Inference and Results Display

In [ ]:
# Model Inference: Load unified model and make predictions (UPDATED for one-hot experiments)
import pickle
import os
import torch

# User Input
model_type = input("Enter model type (LR, RF, or LSTM): ").strip().upper()
genotype = input("Enter genotype name: ").strip()
exp_num = int(input("Enter experiment number (1, 2, or 3): ").strip())
sample = input("Enter sample name (e.g., Camelina01): ").strip()

# Load transpiration data for this sample
transp_file = os.path.join(DATA_DIR, f'CamelinaMAGIC{exp_num}.0-DailyTranspiration.csv')
df_transp = helpers.read_transpiration_csv(transp_file, RANDOMIZATION_FILE, exp_num)
df_transp = df_transp[df_transp['Treatment'] == 'DR_100']

# Extract input features (Days 15-25)
sample_data = df_transp[df_transp['Sample'] == sample].sort_values('Days')
input_slice = sample_data[sample_data['Days'].isin(INPUT_WINDOW)]

if len(input_slice) < len(INPUT_WINDOW):
    print(f"❌ Error: Sample {sample} does not have complete input data (Days 15-25).")
else:
    # Build feature vector with IMPROVED encoding: [11 transpiration, 16 genotype one-hot, 3 experiment one-hot]
    transp_features = input_slice['Transpiration'].values
    
    # One-hot encode genotype
    genotype_encoding = [1 if g == genotype else 0 for g in unique_genotypes]
    
    # One-hot encode experiment (NEW: 3 features instead of 1)
    experiment_encoding = [1 if exp == exp_num else 0 for exp in [1, 2, 3]]
    
    # Combine all features: 11 + 16 + 3 = 30 features
    X_sample = np.concatenate([transp_features, genotype_encoding, experiment_encoding])
    X_sample = X_sample.reshape(1, -1)
    
    # Load model and predict based on type
    if model_type == 'LR':
        model_path = os.path.join(MODELS_DIR, 'LR_unified_model.pkl')
        scaler_path = os.path.join(MODELS_DIR, 'LR_unified_scaler.pkl')
        
        if not os.path.exists(model_path):
            print(f"❌ Error: Model not found. Try 'LR', 'RF', or 'LSTM'")
        else:
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
            with open(scaler_path, 'rb') as f:
                scaler = pickle.load(f)
            
            X_sample_scaled = scaler.transform(X_sample)
            y_pred = model.predict(X_sample_scaled)[0]
        
    elif model_type == 'RF':
        model_path = os.path.join(MODELS_DIR, 'RF_unified_model.pkl')
        scaler_path = os.path.join(MODELS_DIR, 'RF_unified_scaler.pkl')
        
        if not os.path.exists(model_path):
            print(f"❌ Error: Model not found at {model_path}")
        else:
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
            with open(scaler_path, 'rb') as f:
                scaler = pickle.load(f)
            
            X_sample_scaled = scaler.transform(X_sample)
            y_pred = model.predict(X_sample_scaled)[0]
        
    elif model_type == 'LSTM':
        model_path = os.path.join(MODELS_DIR, 'LSTM_unified_model.pth')
        
        if not os.path.exists(model_path):
            print(f"❌ Error: Model not found at {model_path}")
        else:
            # Separate sequence and metadata
            X_seq = X_sample[:, :11].reshape(1, 11, 1)
            X_meta = X_sample[:, 11:]  # Now 19 features (16 genotype + 3 experiment)
            
            # Load model (updated with metadata_size=19)
            model = LSTMModel(input_size=1, hidden_size=64, num_layers=2, output_size=1, metadata_size=19)
            model.load_state_dict(torch.load(model_path))
            model.eval()
            
            # Predict
            with torch.no_grad():
                X_seq_t = torch.FloatTensor(X_seq)
                X_meta_t = torch.FloatTensor(X_meta)
                y_pred = model(X_seq_t, X_meta_t).item()
    else:
        print(f"❌ Error: Unknown model type '{model_type}'. Use 'LR', 'RF', or 'LSTM'.")
        y_pred = None
    
    if y_pred is not None:
        # Calculate actual slope from recovery window
        recovery_slice = sample_data[sample_data['Days'].isin(RECOVERY_WINDOW)]
        
        if len(recovery_slice) < 2:
            print(f"❌ Error: Sample {sample} does not have complete recovery data (Days 36-40).")
        else:
            from scipy.stats import linregress
            y_actual = float(linregress(recovery_slice['Days'], recovery_slice['Transpiration']).slope)
            
            # Calculate error metrics
            error = y_actual - y_pred
            abs_error = abs(error)
            percent_error = (abs_error / abs(y_actual) * 100) if y_actual != 0 else float('inf')
            
            # Display results
            print("\n" + "="*70)
            print(f"✓ INFERENCE RESULTS ({model_type} MODEL)")
            print("="*70)
            print(f"Experiment:        {exp_num}")
            print(f"Genotype:          {genotype}")
            print(f"Sample:            {sample}")
            print(f"Input Period:      Days 15-25 (11 timesteps)")
            print(f"Target Period:     Days 36-40 (Recovery)")
            print("-"*70)
            print(f"Predicted Slope:   {y_pred:.6f} g/day")
            print(f"Actual Slope:      {y_actual:.6f} g/day")
            print("-"*70)
            print(f"Absolute Error:    {abs_error:.6f} g/day")
            print(f"Relative Error:    {percent_error:.2f}%")
            print(f"Error Direction:   {'Overestimated' if y_pred > y_actual else 'Underestimated'}")
            print("="*70 + "\n")
